In [ ]:
from pdf2image import convert_from_path
import pytesseract

pdf_path = "book_part.pdf"

pages = convert_from_path(pdf_path)

full_text = ""

for i, page in enumerate(pages):
    text = pytesseract.image_to_string(page, lang="eng")
    full_text += text + "\n"

print(full_text[:2000])

Mugaddima yoki eng avvalgi bob

QURILISH-MONTAJ BRIGADASINING BOSHLIG‘I
MUZAFFAR SHOMURODOV HIKOYASI

SOVUQ XABAR

Payshanba — maosh kuni edi. Peshindan keyin atrofi
taxta devor bilan omonat o‘ralgan qurilish hovlisiga
chang-to‘zon ko‘tarib «ZIL» mashinasi kirib keldi.
Chekkadagi ko‘chma vagoncha oldida voshillab to‘xtadi.
Kabina eshigi ochilib, go‘ltig‘iga gqora sumka qistirgan
kassir giz — Faya sakrab tushdi. Vagoncha zinasiga
pildirab chigib, baqirdi:

— Yigitlar, kelig‘iz, zarplata olig‘iz!

Betonchi Safar aka ayiqdek lapanglab vagoncha
tomonga birinchi bo‘lib yurdi. Ketidan payvandchimiz
Ikrom aka. Uning ketidan Erkin degan takelajchi yigit...

Saraton oftobi ayovsiz qizdirayotgan, oyoq ostida
qum, beton gorishmasining kukunlam sochilib yotgan

«Rost bilan yolg‘onning o‘rtasi — to‘rt enlik», degan gap bor.
Qiziq, nega endi oz emas, ko'p emas, to‘rt enlik? Gap shundaki,
ko‘z bilan quloqning orasi — to‘rt enlik ekan. Eshitganingga emas,
ko‘rganingga ishon... Magsad — shu.

Bu kitobd

In [7]:
with open("book_text.txt","w",encoding="utf-8") as f:
    f.write(full_text)

In [8]:
chunks = [full_text[i:i+800] for i in range(0, len(full_text), 800)]

print("Chunk soni:", len(chunks))
print(chunks[0])

Chunk soni: 8
Mugaddima yoki eng avvalgi bob

QURILISH-MONTAJ BRIGADASINING BOSHLIG‘I
MUZAFFAR SHOMURODOV HIKOYASI

SOVUQ XABAR

Payshanba — maosh kuni edi. Peshindan keyin atrofi
taxta devor bilan omonat o‘ralgan qurilish hovlisiga
chang-to‘zon ko‘tarib «ZIL» mashinasi kirib keldi.
Chekkadagi ko‘chma vagoncha oldida voshillab to‘xtadi.
Kabina eshigi ochilib, go‘ltig‘iga gqora sumka qistirgan
kassir giz — Faya sakrab tushdi. Vagoncha zinasiga
pildirab chigib, baqirdi:

— Yigitlar, kelig‘iz, zarplata olig‘iz!

Betonchi Safar aka ayiqdek lapanglab vagoncha
tomonga birinchi bo‘lib yurdi. Ketidan payvandchimiz
Ikrom aka. Uning ketidan Erkin degan takelajchi yigit...

Saraton oftobi ayovsiz qizdirayotgan, oyoq ostida
qum, beton gorishmasining kukunlam sochilib yotgan

«Rost bilan yolg‘onning o‘rtasi — to‘rt 


In [ ]:
from google import genai

client = genai.Client(api_key="")

In [27]:
from tqdm import tqdm
import json

dataset = []

for chunk in tqdm(chunks):

    prompt = f"""
Siz Uzbek tilida instruction fine-tuning dataset yaratuvchi AI siz.

Quyidagi matndan 3 ta instruction dataset yarating.

Format:
instruction: foydalanuvchi topshirig'i
input: kontekst yoki matn
output: to'g'ri javob

Qoidalar:
- Faqat berilgan matnga asoslaning
- Uzbek tilida yozing
- Natijani faqat JSON formatda qaytaring

Matn:
{chunk}

JSON format:

[
 {{
  "instruction": "...",
  "input": "...",
  "output": "..."
 }},
 {{
  "instruction": "...",
  "input": "...",
  "output": "..."
 }},
 {{
  "instruction": "...",
  "input": "...",
  "output": "..."
 }}
]
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        text = response.text.strip()
        text = text.replace("```json", "").replace("```", "").strip()

        data = json.loads(text)

        for item in data:
            dataset.append(item)

    except Exception as e:
        print("Error:", e)
        print("Response:", text)



100%|██████████| 8/8 [01:29<00:00, 11.23s/it]


In [29]:
# save dataset
with open("uzbek_instruction_dataset.jsonl","w",encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Dataset size:", len(dataset))

Dataset size: 24


In [ ]:
import json

input_file = "dataset/uzbek_instruction_dataset.jsonl"
output_file = "dataset/final_uzbek_instruction_dataset.json"

data = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Converted to JSON:", output_file)
print("Total records:", len(data))

Converted to JSON: final_uzbek_instruction_dataset.json
Total records: 24
